# Low-SNR Failure — Visual Investigation
**উদ্দেশ্য: high SNR beat করা না — low SNR-এ আসলে কী ঘটছে সেটা চোখে দেখা।**

আগের diagnostic (4 test) থেকে জানা গেছে:
- σ/M দিয়ে ceiling ভাঙে না
- Missed source-দের sin(ψ) কম (end-fire, 0°/180° কাছে)
- Sharpening/oracle-argmax কোনোটাই সাহায্য করে না → diffusion বাদ

**এখানে সেই একই জিনিস নিজের চোখে দেখব:**
1. Model-এর raw heatmap output — true position-এ আদৌ কোনো signal আছে কিনা
2. Missed source-এর জায়গায় zoom করে দেখা — peak নেই? নাকি সরে গেছে? নাকি দুর্বল?
3. এই তিনটে category আলাদা করে quantify করা
4. পরে দ্রুত experiment চালানোর জন্য একটা lightweight ResNet

### Kaggle-এ চালাতে:
1. **Add Data** → `inf_model_007_256_resnet.h5` attach করো
2. **Run All**

In [ ]:
# Cell 1 — Setup
import importlib, subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)
    import cv2

import os, math, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import maximum_filter

np.random.seed(42)
print('✅ Ready')

In [ ]:
# Cell 2 — Physics + Data Generator (paper-exact, self-contained)
class _H(np.ndarray):
    @property
    def H(self): return self.conj().transpose()

def ev(n, angle):
    return ((1/np.sqrt(n)) * np.exp(-1j*np.pi*np.cos(angle)*np.arange(n))).reshape(-1,1)

def make_F(P, nt):
    phi = np.arccos((1/np.pi)*np.angle(np.exp( 1j*(2*np.pi/P)*np.arange(P))))
    F = np.zeros((nt, P), dtype=complex)
    for i, ph in enumerate(phi): F[:, i] = ev(nt, ph).ravel()
    return F

def make_W(Q, nr):
    phi = np.arccos((1/np.pi)*np.angle(np.exp(-1j*(2*np.pi/Q)*np.arange(Q))))
    W = np.zeros((nr, Q), dtype=complex)
    for i, ph in enumerate(phi): W[:, i] = ev(nr, ph).ravel()
    return W

def gen_channel(nr, nt, phi_l, psi_l, alpha_l):
    H = np.zeros((nr, nt), dtype=complex)
    for a, phi, psi in zip(alpha_l, phi_l, psi_l):
        H += a * (ev(nr, psi) * ev(nt, phi).view(_H).H)
    return np.sqrt(nt * nr) * H

def gen_points(L, delta=np.pi/6, max_try=20000):
    pts = []
    for _ in range(max_try):
        if len(pts) == L: break
        x, y = np.random.uniform(0, np.pi), np.random.uniform(0, np.pi)
        if all(math.hypot(x-p[0], y-p[1]) >= delta for p in pts):
            pts.append((x, y))
    if len(pts) < L:
        raise RuntimeError(f'Cannot place {L} points with delta={delta:.3f}')
    return pts

def gen_gt(phi_l, psi_l, M=256, sigma=0.07):
    op = np.mod( np.pi*np.cos(phi_l), 2*np.pi)
    oq = np.mod(-np.pi*np.cos(psi_l), 2*np.pi)
    margin = 3 * sigma
    ax = np.linspace(-margin, 2*np.pi+margin, M, endpoint=False)
    Wp, Wq = np.meshgrid(ax, ax)
    coeff = 1 / (2*np.pi*sigma**2)
    G = sum(coeff * np.exp(-((Wp-o)**2 + (Wq-q)**2)/(2*sigma**2))
            for o, q in zip(op, oq))
    return G.astype(np.float32)

def angles_to_pixel(psi, phi, M=256, sigma=0.07):
    """Where SHOULD the true blob be, in (row, col) pixel coords -- inverse of gen_gt's mapping."""
    op = np.mod( np.pi*np.cos(phi), 2*np.pi)   # column-axis coordinate
    oq = np.mod(-np.pi*np.cos(psi), 2*np.pi)   # row-axis coordinate
    margin = 3*sigma; ext = 2*np.pi + 2*margin
    col = (op + margin) / ext * M
    row = (oq + margin) / ext * M
    return row, col

def data_generation(Training=True, condition=None, sigma=0.07, M=256):
    import random as pyrandom
    while True:
        if Training:
            L   = np.random.randint(1, 10)
            SNR = np.random.randint(-15, 25)
            P   = pyrandom.choice([16, 32])
            nt  = 16 if P == 16 else pyrandom.choice([16, 32])
        else:
            L, SNR, P, nt = condition

        Q, nr = P, nt
        F = make_F(P, nt)
        W = make_W(Q, nr)

        alpha = (np.sqrt(1/L)/np.sqrt(2)) * (np.random.randn(L) + 1j*np.random.randn(L))
        alpha = alpha[np.argsort(-np.abs(alpha))]

        pts   = gen_points(L)
        phi_l = np.array([p[0] for p in pts])
        psi_l = np.array([p[1] for p in pts])

        H = gen_channel(nr, nt, phi_l, psi_l, alpha)
        var = 10**(-SNR/10); s = np.sqrt(var/2)
        Z = s * (np.random.randn(Q, P) + 1j*np.random.randn(Q, P))
        Y = (W.view(_H).H @ H) @ F + Z

        zoom = 4 if P == 16 else 2
        import scipy.ndimage
        data = np.stack([
            scipy.ndimage.zoom(Y.real, zoom, order=0),
            scipy.ndimage.zoom(Y.imag, zoom, order=0)
        ], axis=-1).astype(np.float32)

        if Training:
            gt = gen_gt(phi_l, psi_l, M, sigma)[..., np.newaxis]
            yield data, gt
        else:
            yield data, np.stack([psi_l, phi_l]).astype(np.float32)

print('✅ Physics + data generator ready')

In [ ]:
# Cell 3 — Evaluation utilities
def get_detector():
    p = cv2.SimpleBlobDetector_Params()
    p.filterByColor = True; p.blobColor = 255
    p.minThreshold  = 0;    p.maxThreshold = 255
    p.filterByArea  = True; p.minArea = 1; p.maxArea = 1000
    p.filterByCircularity = p.filterByConvexity = p.filterByInertia = False
    return cv2.SimpleBlobDetector_create(p)

DETECTOR = get_detector()

def get_peaks(pred2d, L):
    img = cv2.normalize(pred2d, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    kps = DETECTOR.detect(img)
    if not kps: return np.zeros((0, 2))
    coords = np.array([k.pt for k in kps])
    amps   = np.array([img[min(int(round(k.pt[1])),img.shape[0]-1),
                           min(int(round(k.pt[0])),img.shape[1]-1)] for k in kps])
    return coords[np.argsort(-amps)[:L]]

def peaks2angles(peaks, sigma=0.07, M=256):
    if len(peaks) == 0: return np.array([]), np.array([])
    margin = 3 * sigma; ext = 2*np.pi + 2*margin
    f = -margin + (peaks.T / M) * ext
    f = np.where(f > np.pi, f - 2*np.pi, f)
    psi = np.arccos(np.clip(-f[1]/np.pi, -1, 1))
    phi = np.arccos(np.clip( f[0]/np.pi, -1, 1))
    return psi, phi

def match_hits_misses(est_psi, est_phi, feat, max_deg=1.0):
    """Returns per-source list of dicts: {hit, dpsi, dphi, true_psi, true_phi}"""
    L = feat.shape[-1]
    true_psi, true_phi = feat[0], feat[1]
    out = []
    if len(est_psi) < L:
        for i in range(L):
            out.append(dict(hit=False, dpsi=None, dphi=None,
                            true_psi=true_psi[i], true_phi=true_phi[i]))
        return out
    est_psi = est_psi[:L]; est_phi = est_phi[:L]
    gt = np.stack([true_psi, true_phi], 1)
    es = np.stack([est_psi, est_phi], 1)
    d = np.linalg.norm(gt[:,None]-es[None], axis=2)
    r, c = linear_sum_assignment(d)
    for i, j in zip(r, c):
        dpsi = np.degrees(np.angle(np.exp(1j*gt[i,0])*np.exp(-1j*es[j,0])))
        dphi = np.degrees(np.angle(np.exp(1j*gt[i,1])*np.exp(-1j*es[j,1])))
        hit = abs(dpsi) <= max_deg and abs(dphi) <= max_deg
        out.append(dict(hit=hit, dpsi=dpsi, dphi=dphi,
                        true_psi=gt[i,0], true_phi=gt[i,1]))
    return out

print('✅ Evaluation utilities ready')

In [ ]:
# Cell 4 — Load pretrained ResNet
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose
from tensorflow.keras.models import Model

def find_weights(filename='inf_model_007_256_resnet.h5'):
    for root in ['/kaggle/input', '/kaggle/working', '.', '/content']:
        if not os.path.isdir(root): continue
        for dp, _, files in os.walk(root):
            if filename in files: return os.path.join(dp, filename)
    return None

def build_plain_ResNet(n_blocks=64, filters=12):
    def res_conv(x, f):
        skip = x
        x = Conv2D(f,5,padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Conv2D(f,5,padding='same')(x); x = BatchNormalization()(x)
        x = Add()([x, skip]); x = Activation('relu')(x)
        return x
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(filters, 5, strides=2, padding='same')(x_in)
    for _ in range(n_blocks): x = res_conv(x, filters)
    x = Conv2DTranspose(1, 5, strides=2, padding='same')(x)
    return Model(x_in, x, name='PlainResNet-64b')

WEIGHTS_PATH = find_weights()
print(f'Weights: {WEIGHTS_PATH}')
assert WEIGHTS_PATH is not None, 'Kaggle-এ inf_model_007_256_resnet.h5 attach করো'

model = build_plain_ResNet(n_blocks=64, filters=12)
model.load_weights(WEIGHTS_PATH)
print(f'✅ Loaded — {model.count_params():,} params')

In [ ]:
# Cell 5 — Collect low-SNR examples: categorize hit / near-endfire-miss / other-miss
np.random.seed(11)
SNR_TARGET = -5      # ← এখানে বদলাতে পারো (-10, -5, 0)
L = 3
N_SCAN = 400

examples = {'hit': [], 'endfire_miss': [], 'other_miss': []}

gen = data_generation(Training=False, condition=(L, SNR_TARGET, 16, 16))
for _ in range(N_SCAN):
    data, feat = next(gen)
    pred = model(tf.expand_dims(data, 0), training=False)[0,:,:,0].numpy()
    pk = get_peaks(pred, L)
    psi_e, phi_e = peaks2angles(pk)
    results = match_hits_misses(psi_e, phi_e, feat)

    for r in results:
        sinpsi = np.sin(r['true_psi'])
        entry = dict(data=data, pred=pred, feat=feat, **r, sinpsi=sinpsi)
        if r['hit']:
            if len(examples['hit']) < 4: examples['hit'].append(entry)
        else:
            if sinpsi < 0.35:
                if len(examples['endfire_miss']) < 4: examples['endfire_miss'].append(entry)
            else:
                if len(examples['other_miss']) < 4: examples['other_miss'].append(entry)

print(f'SNR = {SNR_TARGET} dB  |  scanned {N_SCAN} samples')
for k, v in examples.items():
    print(f'  {k}: {len(v)} examples collected')
if len(examples['other_miss']) == 0:
    print('\n  ⚠️ কোনো non-endfire miss পাওয়া যায়নি — সব miss ই end-fire অঞ্চলে (Test 3-এর সাথে মিলছে)')

In [ ]:
# Cell 6 — VISUAL: full heatmap + true/detected position overlay
def plot_examples(cat_name, entries, max_show=3):
    entries = entries[:max_show]
    if not entries:
        print(f'({cat_name}: কোনো example নেই)')
        return
    fig, axes = plt.subplots(len(entries), 2, figsize=(9, 4.2*len(entries)))
    if len(entries) == 1: axes = axes.reshape(1, 2)
    fig.suptitle(f'Category: {cat_name}', fontsize=13, y=1.01, fontweight='bold')

    for i, e in enumerate(entries):
        # true positions in pixel coords
        L_ = e['feat'].shape[-1]
        true_rows, true_cols = angles_to_pixel(e['feat'][0], e['feat'][1])

        # left: raw prediction heatmap
        ax = axes[i, 0]
        im = ax.imshow(e['pred'], cmap='viridis', origin='lower')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.scatter(true_cols, true_rows, marker='x', c='red', s=80, label='true', linewidths=2)
        my_row, my_col = angles_to_pixel(e['true_psi'], e['true_phi'])
        ax.scatter([my_col], [my_row], marker='o', facecolors='none', edgecolors='yellow', s=200, linewidths=2, label='this source')
        ax.set_title(f"Model output — {'HIT' if e['hit'] else 'MISS'}  (ψ={np.degrees(e['true_psi']):.0f}°, sinψ={e['sinpsi']:.2f})")
        ax.legend(loc='upper right', fontsize=7)

        # right: zoomed crop around the true location (the money shot)
        ax = axes[i, 1]
        r0, c0 = int(my_row), int(my_col)
        pad = 25
        r1, r2 = max(0, r0-pad), min(256, r0+pad)
        c1, c2 = max(0, c0-pad), min(256, c0+pad)
        crop = e['pred'][r1:r2, c1:c2]
        im = ax.imshow(crop, cmap='viridis', origin='lower',
                       extent=[c1,c2,r1,r2])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.scatter([my_col], [my_row], marker='+', c='red', s=200, linewidths=2)
        ax.set_title(f'Zoomed crop around true location (±{pad}px)')

    plt.tight_layout()
    plt.show()

print('Plotting HIT examples (baseline — for comparison):')
plot_examples('HIT (successful detection)', examples['hit'])

print('\nPlotting END-FIRE MISS examples (ψ near 0°/180°):')
plot_examples('MISS — end-fire (sinψ < 0.35)', examples['endfire_miss'])

print('\nPlotting OTHER MISS examples (if any):')
plot_examples('MISS — mid-range angle', examples['other_miss'])

In [ ]:
# Cell 7 — Quantify: is the peak MISSING, WEAK, or SHIFTED at the true location?
# For each missed source: read the model's raw value exactly at the true pixel location
# vs. the model's global max (contrast). Also measure how far the NEAREST local max
# actually is from the true location (shift distance in pixels).

def analyze_miss(entry):
    pred = entry['pred']
    my_row, my_col = angles_to_pixel(entry['true_psi'], entry['true_phi'])
    r0, c0 = int(round(my_row)), int(round(my_col))
    r0 = np.clip(r0, 0, 255); c0 = np.clip(c0, 0, 255)

    val_at_true = pred[r0, c0]
    global_max   = pred.max()
    ratio = val_at_true / (global_max + 1e-9)

    # nearest strong local max within a radius
    pad = 40
    r1,r2 = max(0,r0-pad), min(256,r0+pad)
    c1,c2 = max(0,c0-pad), min(256,c0+pad)
    crop = pred[r1:r2, c1:c2]
    mx = maximum_filter(crop, size=5, mode='nearest')
    loc_idx = np.argwhere(crop == mx)
    if len(loc_idx):
        vals = crop[loc_idx[:,0], loc_idx[:,1]]
        best = loc_idx[np.argmax(vals)]
        shift_px = math.hypot(best[0]-(r0-r1), best[1]-(c0-c1))
    else:
        shift_px = float('nan')

    return ratio, shift_px

print('End-fire miss diagnostics (value-at-true-location / global-max, nearest-peak-shift):')
print('='*70)
for e in examples['endfire_miss']:
    ratio, shift = analyze_miss(e)
    print(f"  ψ={np.degrees(e['true_psi']):.1f}°  sinψ={e['sinpsi']:.3f}  "
          f"→ signal ratio at true loc: {ratio:.3f}   nearest peak shift: {shift:.1f}px")

print()
print('ব্যাখ্যা:')
print('  ratio ≈ 1.0        → true location-এই peak আছে, কিন্তু detector miss করেছে (extraction bug)')
print('  ratio ছোট (<0.3)   → true location-এ signal দুর্বল/অনুপস্থিত (network ব্যর্থ)')
print('  shift বড় (>5px)    → peak সঠিক জায়গায় নেই, পাশে সরে গেছে (Jacobian/localization issue)')

In [ ]:
# Cell 8 — Statistics over many samples: ratio & shift vs sin(psi), across SNR
np.random.seed(21)
L = 3
N_TRIALS = 500
SNRS = [-10, -5, 0, 10, 25]

stat = {snr: {'sinpsi': [], 'ratio': [], 'shift': [], 'hit': []} for snr in SNRS}

for snr in SNRS:
    gen = data_generation(Training=False, condition=(L, snr, 16, 16))
    for _ in range(N_TRIALS):
        data, feat = next(gen)
        pred = model(tf.expand_dims(data, 0), training=False)[0,:,:,0].numpy()
        pk = get_peaks(pred, L)
        psi_e, phi_e = peaks2angles(pk)
        results = match_hits_misses(psi_e, phi_e, feat)
        for r in results:
            entry = dict(pred=pred, true_psi=r['true_psi'], true_phi=r['true_phi'])
            ratio, shift = analyze_miss(entry)
            stat[snr]['sinpsi'].append(np.sin(r['true_psi']))
            stat[snr]['ratio'].append(ratio)
            stat[snr]['shift'].append(shift)
            stat[snr]['hit'].append(r['hit'])
    print(f'SNR={snr}dB done ({N_TRIALS*L} sources scanned)')

fig, axes = plt.subplots(1, len(SNRS), figsize=(4*len(SNRS), 4), sharey=True)
for ax, snr in zip(axes, SNRS):
    s = np.array(stat[snr]['sinpsi']); ratio = np.array(stat[snr]['ratio']); hit = np.array(stat[snr]['hit'])
    ax.scatter(s[hit], ratio[hit], s=10, alpha=0.4, c='green', label='hit')
    ax.scatter(s[~hit], ratio[~hit], s=10, alpha=0.4, c='red', label='miss')
    ax.set_xlabel('sin(ψ)'); ax.set_title(f'SNR={snr}dB')
    ax.axhline(0.3, ls='--', c='gray', lw=0.8)
axes[0].set_ylabel('signal ratio at true location'); axes[0].legend(fontsize=8)
plt.suptitle('Signal strength at TRUE location, vs sin(ψ) — colored by hit/miss', y=1.03)
plt.tight_layout(); plt.show()

print()
for snr in SNRS:
    ratio = np.array(stat[snr]['ratio']); hit = np.array(stat[snr]['hit'])
    miss_ratio = ratio[~hit]
    print(f'SNR={snr:4d}dB   miss-ratio mean={np.nanmean(miss_ratio):.3f}   '
          f'(n_miss={len(miss_ratio)})')

---
## Lightweight ResNet — future দ্রুত experiment-এর জন্য

Cell 7-8 এর ফলাফল দেখে fix ঠিক করার পর, প্রতিবার ৫.৫ ঘণ্টা wait না করে **ছোট ResNet** (block কম) দিয়ে দ্রুত test করা যায়।

নিচের model paper-এর তুলনায় parameter/time কতটা কমে, আর কতটা close থাকে সেটা দেখাবে।

In [ ]:
# Cell 9 — Lightweight ResNet builder (for fast iteration on fixes)
def build_lite_ResNet(n_blocks=16, filters=12):
    def res_conv(x, f):
        skip = x
        x = Conv2D(f,5,padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Conv2D(f,5,padding='same')(x); x = BatchNormalization()(x)
        x = Add()([x, skip]); x = Activation('relu')(x)
        return x
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(filters, 5, strides=2, padding='same')(x_in)
    for _ in range(n_blocks): x = res_conv(x, filters)
    x = Conv2DTranspose(1, 5, strides=2, padding='same')(x)
    return Model(x_in, x, name=f'LiteResNet-{n_blocks}b')

print(f'{"Blocks":>8} {"Params":>10} {"vs full-64":>12} {"est. train (500ep, T4)":>24}')
print('-'*58)
for nb in [8, 16, 24, 32, 64]:
    m = build_lite_ResNet(nb, 12)
    p = m.count_params()
    frac = p / 469393
    # rough per-step time scales roughly with block count (compute-bound conv stack)
    est_hours = 5.5 * (nb/64)
    print(f'{nb:>8} {p:>10,} {frac:>11.1%} {est_hours:>22.1f}h')
    del m

print()
print('সুপারিশ: 16-block variant দিয়ে শুরু করো (~1.4h/500-epoch run)।')
print('যদি tren (Pd/RMSE shape, end-fire miss pattern) paper/64-block-এর মতো দেখায়,')
print('তাহলে সব fix ওই ছোট model-এ দ্রুত validate করে, শেষে সেরা fix-টা 64-block-এ scale up করো।')